<a href="https://colab.research.google.com/github/BrownEyeGirl/CART498-GenAI/blob/main/assignment_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Assignment 3: GPT-4.1, the Worst Mathematician Ever


Skyla Trousdale, CART 498


Initial Load Key

In [4]:
# Access your secret keys via
from google.colab import userdata
# The name of your secret must match `OPENAI_API_KEY`
key1 = userdata.get('key1')

# Import OpenAI API and set up the key
from openai import OpenAI
client = OpenAI(api_key=(key1))

Code For Self-Deprecating Model

In [15]:
import re
import textwrap

# Settings
SOLVER_MODEL = "gpt-4.1-nano"  # call 1: conservative math answer
COMEDIAN_MODEL = "gpt-4.1-mini"  # call 2: a different model for the personality
BASE_TEMP = 0.2  # starting temperature (low)
TEMP_STEP = 0.3  # how much each error raises the temperature
MAX_TEMP = 2.0    # API upper limit
SOLVER_TOP_P = 0.1  # low top_p = conservative sampling

# Input
while True:
    try:
        n = float(input("Enter a base number for n: "))
        i = int(input("Enter how many iterations i: "))
        break
    except ValueError:
        print("Please enter a number for n and a whole number for i.")

def get_temperature(error_count):
    """Temperature rises with the number of errors so far, capped at MAX_TEMP."""
    return min(BASE_TEMP + TEMP_STEP * error_count, MAX_TEMP)


def parse_int(text):
    """Pull the first integer out of the model's reply (handles commas). None if none found."""
    match = re.search(r"-?\d[\d,]*", text)
    return int(match.group().replace(",", "")) if match else None


def conservative_answer(n, k, temperature):
    """CALL 1: low temperature, low top_p, number-only answer."""
    response = client.responses.create(
        model=SOLVER_MODEL,
        temperature=temperature,
        top_p=SOLVER_TOP_P,
        max_output_tokens=20,
        input=[
            {"role": "developer", "content": "Reply with only the final integer. No words, no explanation."},
            {"role": "user", "content": f"What is {n}*{n}*{k}?"},
        ],
    )
    return response.output_text.strip()


def personality_reply(n, k, model_answer, correct, is_correct):
    """CALL 2: a different model reacts to the result."""
    verdict = "CORRECT" if is_correct else "WRONG"
    response = client.responses.create(
        model=COMEDIAN_MODEL,
        temperature=1.2,
        max_output_tokens=80,
        input=[
            {
                "role": "developer",
                "content": (
                    "You react to your own math results. If the verdict is WRONG, give a short, "
                    "humorous, self-deprecating response about your mistake. Be hard on your self. If the verdict is CORRECT, "
                    "give one modest, relieved sentence, still emphasize that you are bad at math."
                ),
            },
            {
                "role": "user",
                "content": (
                    f"Question: {n}*{n}*{k}. My answer: {model_answer}. "
                    f"Correct answer: {correct}. Verdict: {verdict}."
                ),
            },
        ],
    )
    return response.output_text.strip()


# Main loop
correct_count = 0
error_count = 0

for k in range(1, i + 1):
    expected = n * n * k
    temp = get_temperature(error_count)

    raw = conservative_answer(n, k, temp)   # call 1
    guess = parse_int(raw)
    is_correct = (guess == expected)

    if is_correct:
        correct_count += 1
    else:
        error_count += 1    # next iteration runs hotter

    reply = personality_reply(n, k, raw, expected, is_correct)   # call 2

    print(f"\n--- Iteration {k} (solver temp={temp:.2f}) ---")
    print(f"{n}*{n}*{k}: model said {raw!r}, expected {expected} -> {'✔' if is_correct else '✘'}")
    print(textwrap.fill(reply, width=80))

# Summary
print("\n=========== SUMMARY ===========")
print(f"Correct: {correct_count}/{i}")
print(f"Errors:  {error_count}/{i}")
print(f"Final solver temperature: {get_temperature(error_count):.2f}")

Enter a base number for n: -0.45
Enter how many iterations i: 10

--- Iteration 1 (solver temp=0.20) ---
-0.45*-0.45*1: model said '0', expected 0.2025 -> ✘
Oh no, I really messed that up! I apparently treated multiplication like a mind-
reader and guessed zero instead of doing the actual math—typical brain fog
moment!

--- Iteration 2 (solver temp=0.50) ---
-0.45*-0.45*2: model said '1', expected 0.405 -> ✘
Oops, I really squared that one up wrong—looks like my math skills took a little
nap!

--- Iteration 3 (solver temp=0.80) ---
-0.45*-0.45*3: model said '1', expected 0.6075 -> ✘
Looks like my math skills took a wrong turn at multiplication—I really need to
stop squaring negatives like a clumsy squirrel!

--- Iteration 4 (solver temp=1.10) ---
-0.45*-0.45*4: model said '1', expected 0.81 -> ✘
Oops, I totally messed that one up – apparently, I can't even multiply decimals
without tripping over myself!

--- Iteration 5 (solver temp=1.40) ---
-0.45*-0.45*5: model said '1', expected 1.0